# Demonstração do pipeline RAG — Voice of Customer (Olist)

Este notebook percorre, etapa por etapa, o caminho de uma pergunta até a resposta, usando os
mesmos módulos da aplicação (`src/voc_rag`). Pré-requisito: base e índices já construídos
(`python -m voc_rag.data_prep` e `python -m voc_rag.indexing`).

In [ ]:
import sys, json
sys.path.insert(0, "../src")
import pandas as pd
pd.set_option("display.max_colwidth", 110)

from voc_rag import config
from voc_rag.query import plan_query
from voc_rag.pipeline import VoCPipeline
from voc_rag.generation import build_context

## 1. Base de conhecimento
Resumo do que `data_prep.py` produziu, incluindo as verificações de integridade.

In [ ]:
report = json.loads(config.DATA_REPORT_FILE.read_text())
print("Avaliações no CSV:", report["total_reviews"], "| com texto:", report["reviews_with_text"],
      "| documentos finais:", report["documents"])
pd.Series(report["integrity"], name="ocorrências").to_frame()

In [ ]:
docs = pd.read_parquet(config.DOCUMENTS_FILE)
docs[["text", "review_score", "purchase_date", "category", "delivery_status", "delay_days"]].sample(5, random_state=7)

In [ ]:
# Relação entre situação da entrega e nota (motiva o uso do metadado de entrega)
docs.groupby("delivery_status")["review_score"].agg(["count", "mean"]).round(2)

## 2. Transformação da pergunta
`plan_query` produz o plano de consulta: intenção, filtros inferidos, expansão léxica e classificação ampla × focada.

In [ ]:
pergunta = "O que os clientes relatam sobre problemas relacionados à entrega?"
plan = plan_query(pergunta)
plan.as_dict()

## 3. Recuperação
Busca híbrida (densa + BM25) com filtros, fusão RRF, verificação de suporte e MMR.

In [ ]:
pipe = VoCPipeline()   # usa o provedor configurado no .env; sem chave, cai no modo extrativo
resp = pipe.ask(pergunta)
print("modo:", resp.retrieval.mode, "| população filtrada:", resp.retrieval.population, "|", resp.retrieval.reason)
pd.DataFrame([{"ref": e.ref, "nota": e.score, "entrega": e.delivery_status, "sim_densa": e.dense_sim,
               "rank_bm25": e.bm25_rank, "texto": e.text} for e in resp.evidences])

## 4. Contexto enviado ao modelo de linguagem

In [ ]:
print(resp.context)

## 5. Resposta gerada e validação das citações

In [ ]:
print("gerador:", resp.generator, "\n")
print(resp.answer)
print("\nvalidação:", {k: v for k, v in resp.citation_check.items() if k != "answer"})

## 6. Pergunta ampla: decomposição por tema
Sem termo de conteúdo na pergunta, as evidências são buscadas por tema: duas para cada um dos quatro temas mais frequentes na população filtrada.

In [ ]:
resp2 = pipe.ask("Quais são os principais problemas relatados pelos clientes em suas avaliações?")
print("modo:", resp2.retrieval.mode, "| filtros:", resp2.plan.filters.describe())
display(pd.DataFrame(resp2.retrieval.theme_stats_population))
pd.DataFrame([{"ref": e.ref, "tema": e.subquery, "nota": e.score, "texto": e.text} for e in resp2.evidences])

## 7. Perguntas sem evidência suficiente
A recusa ocorre antes da chamada ao modelo de linguagem.

In [ ]:
for q in ["Qual a capital da França?", "Qual foi o faturamento da Olist em 2018?"]:
    r = pipe.ask(q)
    print(q, "->", r.generator, "\n  ", r.answer, "\n")

## 8. Filtros de metadados
Filtros explícitos (ou inferidos) são aplicados dentro da busca vetorial e do BM25.

In [ ]:
resp3 = pipe.ask("O que os clientes mais elogiam em produtos de beleza?")
print(resp3.plan.filters.describe(), "| população:", resp3.retrieval.population)
pd.DataFrame([{"ref": e.ref, "nota": e.score, "categoria": e.category, "texto": e.text} for e in resp3.evidences]).head(6)